# Intro

In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import datetime as dt

In [2]:
folder_path = r'C:\Users\santi\Documents\GitHub\Data Science Portfolio\Agricultura\Datasets'

csv_files = []
dfs = []

for f in os.listdir(folder_path):
  if f.endswith('.csv'):
    csv_files.append(f)

for filename in csv_files:
  file_path = os.path.join(folder_path, filename)
  temporary_df = pd.read_csv(file_path, encoding = 'ISO-8859-1')
  dfs.append(temporary_df)

df = pd.concat(dfs, ignore_index = True)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118549 entries, 0 to 118548
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   cultivo_nombre           118549 non-null  object 
 1   anio                     118549 non-null  int64  
 2   campania                 118549 non-null  object 
 3   provincia_nombre         118549 non-null  object 
 4   provincia_id             118549 non-null  int64  
 5   departamento_nombre      118079 non-null  object 
 6   departamento_id          103542 non-null  float64
 7   superficie_sembrada_ha   118549 non-null  float64
 8   superficie_cosechada_ha  106386 non-null  float64
 9   produccion_tm            96131 non-null   float64
 10  rendimiento_kgxha        106503 non-null  float64
 11  departamnto_id           14354 non-null   float64
 12  produccion_tn            10395 non-null   float64
dtypes: float64(7), int64(2), object(4)
memory usage: 11.8+ MB


# Limpieza

Reemplazamos todas las letras con tilde por la misma letra sin tilde para evitar futuros errores

Del mismo modo convertimos todas las letras minúscula

In [3]:
from unidecode import unidecode

for col in ['cultivo_nombre', 'provincia_nombre', 'departamento_nombre']:
    df[col] = df[col].apply(lambda x: unidecode(str(x)).lower())

Parece haber 2 columnas con nombre muy similares:

*   `departamento_id` con `departamnto_id`
*   `produccion_tn` con `produccion_tm`


SI se trata de una confusión de los autores del dataset al ponerle el nombre a las columnas de algunos datasets entonces se cumpliría lo siguiente:

*   Donde la primer columna tenga valores, la otra tiene nulos y viceversa
*   El punto anterior se exceptua cuando son valores nulos en ambas columnas (dato que realmente no fue capturado). La sumatoria de los datos 'válidos' de cada columna debería ser igual a la cantidad de filas de todo el dataset.



In [4]:
condition_1 = df['departamento_id'].notna() & df['departamnto_id'].isna()
condition_2 = df['departamento_id'].isna() & df['departamnto_id'].notna()
df['valid_condition'] = condition_1 | condition_2
df['valid_condition'].value_counts()

valid_condition
True     117896
False       653
Name: count, dtype: int64

In [5]:
df['both_na'] = df['departamento_id'].isna() & df['departamnto_id'].isna()
df['both_na'].value_counts()

both_na
False    117896
True        653
Name: count, dtype: int64

In [6]:
df.shape[0] == (df['valid_condition'].value_counts().iloc[0] + df['both_na'].value_counts().iloc[1])

True

Se cumple lo explicado antes para `departamento_id` y `departamnto_id` por ende completo una columna con los valores de la otra y paso a borrar las columnas auxiliares que ya no son necesarias

In [7]:
df['departamento_id_combined'] = df['departamento_id'].fillna(df['departamnto_id'])

In [8]:
df['departamento_id_combined'].isna().value_counts()

departamento_id_combined
False    117896
True        653
Name: count, dtype: int64

In [9]:
df.drop(['departamento_id', 'departamnto_id', 'valid_condition', 'both_na'], axis=1, inplace=True)
df.rename(columns={'departamento_id_combined': 'departamento_id'}, inplace=True)

In [10]:
condition_4 = df['produccion_tn'].notna() & df['produccion_tm'].isna()
condition_5 = df['produccion_tn'].isna() & df['produccion_tm'].notna()
df['valid_condition'] = condition_4 | condition_5
df['valid_condition'].value_counts()

valid_condition
True     106526
False     12023
Name: count, dtype: int64

In [11]:
condition_6 = df['produccion_tn'].isna() & df['produccion_tm'].isna()
df['both_na'] = condition_6
df['both_na'].value_counts()

both_na
False    106526
True      12023
Name: count, dtype: int64

In [12]:
df.shape[0] == (df['valid_condition'].value_counts().iloc[0] + df['both_na'].value_counts().iloc[1])

True

Se cumple lo explicado antes para `produccion_tn` y `produccion_tm` por ende completo una columna con los valores de la otra y paso a borrar las columnas auxiliares que ya no son necesarias

In [13]:
df['produccion_tn_combined'] = df['produccion_tn'].fillna(df['produccion_tm'])

In [14]:
df['produccion_tn_combined'].isna().value_counts()

produccion_tn_combined
False    106526
True      12023
Name: count, dtype: int64

In [15]:
df.drop(['produccion_tn', 'produccion_tm', 'valid_condition', 'both_na'], axis=1, inplace=True)
df.rename(columns={'produccion_tn_combined': 'produccion_tn'}, inplace=True)

In [16]:
df['cultivo_nombre'].unique()

array(['arroz', 'avena', 'cebada cervecera', 'cebada forrajera',
       'centeno', 'girasol', 'maiz', 'mani', 'soja', 'mijo'], dtype=object)

Estandarizo los nombres de los cultivos

In [17]:
crop_change = {'Arroz':'arroz', 'maíz':'maiz', 'girasol':'girasol', 'maní':'mani',
               'manï':'mani', 'mijo':'sorgo', 'avena':'avena',
               'Cebada cervecera':'cebada cervecera', 'centeno':'centeno',
               'Cebada forrajera':'cebada forrajera', 'soja':'soja'}

df['cultivo_nombre'] = df['cultivo_nombre'].replace(crop_change)

In [18]:
unique_values = {}

for col in df.columns:
  unique_values[col] = df[col].unique()

In [19]:
features_to_delete = []

for col, value in unique_values.items():
  if len(value) < 2:
    print(f'{col}: {value} \n')
    features_to_delete.append(col)

Todas las columnas tienen más de 1 valor. Esta verificación se hace porque las columnas con 1 solo valor no aportan nada al modelo y consumen capacidad de procesamiento

In [20]:
for col, value in unique_values.items():
  if len(value) < 10:
    print(f'{col}: {value} \n')

Todas las columnas tienen más de 10 valores

In [21]:
for col, value in unique_values.items():
  q = len(value)
  print(f'{col}: {q}')

cultivo_nombre: 10
anio: 100
campania: 510
provincia_nombre: 25
provincia_id: 24
departamento_nombre: 451
superficie_sembrada_ha: 5471
superficie_cosechada_ha: 6392
rendimiento_kgxha: 5723
departamento_id: 533
produccion_tn: 13535


1.   Tener 100 valores de `anio` y más de 500 de `campania` es raro
2.   Misma observación para `departamento_nombre` y `departamento_id`
3.   Misma observación para `provincia_nombre` y `provincia_id`

`Las observaciones 2 y 3, se resuelven con la tabla extra de ubicaciones y reseteando los id de cada ubicación`

In [22]:
round(df.isnull().sum()*100/df.shape[0], 2).sort_values(ascending=False)

superficie_cosechada_ha    10.26
rendimiento_kgxha          10.16
produccion_tn              10.14
departamento_id             0.55
cultivo_nombre              0.00
anio                        0.00
campania                    0.00
provincia_nombre            0.00
provincia_id                0.00
departamento_nombre         0.00
superficie_sembrada_ha      0.00
dtype: float64

`La cantidad de valores nulos no es mucha (en el caso de los ids nulos no es relevante ya que crearemos nuestros propios ids para provincias, cultivos y departamentos)`


`Por otro lado, las variables con mayor % de nulos están muy relacionadas entre sí: superficie_cosechada_ha, produccion_tn, rendimiento_kgxha (esta última surge de la relación entre las otras dos)`

In [23]:
df.produccion_tn[df['superficie_cosechada_ha'].isnull()].isnull().value_counts()

produccion_tn
True     11986
False      177
Name: count, dtype: int64

`Acá vemos que de todos los registros donde la superficie cosechada es nulo, la de producción tmb es nula en practicamente todas`

`En todas aquellas columnas donde el rendimiento es cero, la superficie cosechada o la producción deberían ser cero`

In [24]:
df['rendimiento_kgxha'][df['superficie_cosechada_ha'].isnull() & df['produccion_tn'].isnull()].isnull().value_counts()

rendimiento_kgxha
True     11981
False        5
Name: count, dtype: int64

In [25]:
df['rendimiento_kgxha'][df['superficie_cosechada_ha'].isnull() & df['produccion_tn'].isnull()].value_counts()

rendimiento_kgxha
0.0       3
800.0     1
1000.0    1
Name: count, dtype: int64

`Es decir que de aquellos registros donde la superficie cosechada y la producción son nulas, solo 5 tienen un dato de rendimiento. Y de esos 5, en 3 ocasiones es CERO`

`Por ende, con los datos disponibles no podemos completar esas columnas utilizando la información de las otras`

In [26]:
df['row_to_delete'] = df['superficie_cosechada_ha'].isnull() | df['produccion_tn'].isnull() | df['rendimiento_kgxha'].isnull()
df = df[df['row_to_delete'] == False]

In [27]:
round(df.isnull().sum()*100/df.shape[0], 2).sort_values(ascending=False)

departamento_id            0.49
cultivo_nombre             0.00
anio                       0.00
campania                   0.00
provincia_nombre           0.00
provincia_id               0.00
departamento_nombre        0.00
superficie_sembrada_ha     0.00
superficie_cosechada_ha    0.00
rendimiento_kgxha          0.00
produccion_tn              0.00
row_to_delete              0.00
dtype: float64

In [28]:
df.drop('row_to_delete', axis=1, inplace=True)

In [29]:
df = df.astype({'provincia_id': 'object',
                'departamento_id': 'object'})

In [30]:
types = list(df.dtypes)
unique_types = list(set(types))
unique_types

[dtype('int64'), dtype('float64'), dtype('O')]

In [31]:
numeric_features = list(df.select_dtypes(include=['int', 'float']).columns)
categoric_features = list(df.select_dtypes(include=['object']).columns)

In [32]:
print(numeric_features)
print(categoric_features)

['anio', 'superficie_sembrada_ha', 'superficie_cosechada_ha', 'rendimiento_kgxha', 'produccion_tn']
['cultivo_nombre', 'campania', 'provincia_nombre', 'provincia_id', 'departamento_nombre', 'departamento_id']


In [33]:
df_departamentos = df[['provincia_nombre', 'departamento_nombre']]
df_departamentos.drop_duplicates(inplace=True)
df_departamentos.sort_values(by=['provincia_nombre', 'departamento_nombre'],
                             ascending=[True,True], inplace=True)
df_departamentos = df_departamentos.reset_index(drop=True)
df_departamentos = df_departamentos.reset_index()
df_departamentos.rename(columns={'provincia_nombre':'provincia',
                                 'departamento_nombre':'departamento',
                                 'index': 'id_ubicacion'},
                        inplace=True)
df_departamentos['ubicacion_completa'] = df_departamentos['departamento'].astype(str) + ', ' + df_departamentos['provincia'].astype(str)
df_departamentos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 582 entries, 0 to 581
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id_ubicacion        582 non-null    int64 
 1   provincia           582 non-null    object
 2   departamento        582 non-null    object
 3   ubicacion_completa  582 non-null    object
dtypes: int64(1), object(3)
memory usage: 18.3+ KB


C:\Users\santi\AppData\Local\Temp\ipykernel_11972\497800207.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_departamentos.drop_duplicates(inplace=True)
C:\Users\santi\AppData\Local\Temp\ipykernel_11972\497800207.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_departamentos.sort_values(by=['provincia_nombre', 'departamento_nombre'],


In [34]:
df_departamentos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 582 entries, 0 to 581
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id_ubicacion        582 non-null    int64 
 1   provincia           582 non-null    object
 2   departamento        582 non-null    object
 3   ubicacion_completa  582 non-null    object
dtypes: int64(1), object(3)
memory usage: 18.3+ KB


In [35]:
df['ubicacion_completa'] = df['departamento_nombre'].astype(str) + ', ' + df['provincia_nombre'].astype(str)

In [36]:
df_cultivos = df[['cultivo_nombre']]
df_cultivos.drop_duplicates(inplace=True)
df_cultivos.sort_values(by='cultivo_nombre',
                        ascending=True,
                        inplace=True)
df_cultivos = df_cultivos.reset_index(drop=True)
df_cultivos = df_cultivos.reset_index()
df_cultivos.rename(columns={'cultivo_nombre': 'cultivo',
                            'index': 'id_cultivo'},
                   inplace = True)

df_cultivos = df_cultivos.reset_index(drop=True)
df_cultivos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id_cultivo  10 non-null     int64 
 1   cultivo     10 non-null     object
dtypes: int64(1), object(1)
memory usage: 288.0+ bytes


C:\Users\santi\AppData\Local\Temp\ipykernel_11972\1355702451.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cultivos.drop_duplicates(inplace=True)
C:\Users\santi\AppData\Local\Temp\ipykernel_11972\1355702451.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cultivos.sort_values(by='cultivo_nombre',


In [37]:
df = df.merge(df_departamentos[['ubicacion_completa', 'id_ubicacion']],
              on = 'ubicacion_completa',
              how = 'left')
df['id_ubicacion'].isnull().value_counts()

id_ubicacion
False    106315
Name: count, dtype: int64

In [38]:
df = df.merge(df_cultivos[['cultivo', 'id_cultivo']],
              left_on = 'cultivo_nombre',
              right_on = 'cultivo',
              how = 'left')

In [39]:
df['id_cultivo'].isnull().value_counts()

id_cultivo
False    106315
Name: count, dtype: int64

In [40]:
df.drop(['cultivo_nombre', 'provincia_id', 'departamento_id', 'ubicacion_completa'],
        axis=1,
        inplace=True)
df.head()

,anio,campania,provincia_nombre,departamento_nombre,superficie_sembrada_ha,superficie_cosechada_ha,rendimiento_kgxha,produccion_tn,id_ubicacion,cultivo,id_cultivo
0,1924,1924/1925,corrientes,santo tome,89.0,89.0,1000.0,89.0,252,arroz,0
1,1924,1924/1925,formosa,pilagas,1.0,1.0,1000.0,1.0,282,arroz,0
2,1924,1924/1925,jujuy,el carmen,100.0,100.0,2000.0,200.0,289,arroz,0
3,1924,1924/1925,jujuy,san pedro,250.0,250.0,2280.0,570.0,300,arroz,0
4,1924,1924/1925,jujuy,santa barbara,150.0,150.0,2200.0,330.0,301,arroz,0


In [41]:
df = df[['cultivo',	'id_cultivo', 'anio',	'campania', 'id_ubicacion',
         'provincia_nombre', 'departamento_nombre', 'superficie_sembrada_ha',
         'superficie_cosechada_ha', 'produccion_tn',	'rendimiento_kgxha']]

# Guardar data

In [42]:
excel = r'C:\Users\santi\Documents\GitHub\Data Science Portfolio\Agricultura\dataframes_output.xlsx'
with pd.ExcelWriter(excel, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='main', index=False)
    df_cultivos.to_excel(writer, sheet_name='cultivos', index=False)
    df_departamentos.to_excel(writer, sheet_name='departamentos', index=False)
    print('\ndf guardados')


df guardados


In [43]:
excel = r'C:\Users\santi\Documents\GitHub\Data Science Portfolio\Agricultura\departamentos.xlsx'
with pd.ExcelWriter(excel, engine='openpyxl') as writer:
    df_departamentos.to_excel(writer, sheet_name='departamentos', index=False)
    print('\ndf guardados')


df guardados


# EDA

In [ ]:
colours = ['#f2a73d', '#f76157', '#f6daab', '#dabd7b', '#f04155', '#ff823a', '#f2f26f', '#fff7bd', '#95cfb7', '#f40034',
           '#07f9a2', '#09c184', '#0a8967', '#0c5149', '#0d192b', '#fe1cac', '#820081', '#e4b302', '#e7204e', '#3f2c26']

In [ ]:
axtitle_dict = {'family': 'serif', 'color': 'darkred', 'weight': 'bold', 'size': 12}
axlab_dict = {'family': 'serif', 'color': 'black', 'size': 10}

In [ ]:
count = df['anio'].value_counts()

In [ ]:
ax = sns.lineplot(x=count.index, y=count.values)
ax.set_title('Registros por año', fontdict = axtitle_dict)
ax.set_xlabel('Año', fontdict = axlab_dict)

In [ ]:
prod_anio = df.groupby('anio')['produccion_tn'].sum().reset_index()

In [ ]:
ax = sns.lineplot(x=prod_anio.anio, y=prod_anio.produccion_tn)
ax.set_title('Produccion anual', fontdict=axtitle_dict)
ax.set_xlabel('Año', fontdict=axlab_dict)

In [ ]:
prod_prov = df.groupby('provincia_nombre')['produccion_tn'].sum().reset_index().sort_values(by='produccion_tn', ascending=False).reset_index(drop=True)

In [ ]:
ax = sns.barplot(x=prod_prov.provincia_nombre, y=prod_prov.produccion_tn)
ax.set_title('Producción anual', fontdict=axtitle_dict)
ax.set_xlabel('Provincia', fontdict=axlab_dict)
plt.xticks(rotation=90)
plt.show()

In [ ]:
prod_prov_2 = prod_prov.copy()
prod_prov_2 = prod_prov_2.sort_values(by='produccion_tn', ascending=False)
prod_prov_2 = prod_prov_2.loc[0:10,:].sort_values(by='produccion_tn', ascending=True)

In [ ]:
prod_prov_2['acumulado'] = prod_prov_2['produccion_tn'].cumsum() *100 / prod_prov_2['produccion_tn'].sum()

In [ ]:
prod_prov_2.acumulado

In [ ]:
# Crear el gráfico
fig, ax1 = plt.subplots(figsize=(10, 6))

# Gráfico de barras (valores)
sns.barplot(x=prod_prov_2.provincia_nombre, y=prod_prov_2.produccion_tn, ax=ax1)
ax1.set_ylabel('Produccion [tn]', fontdict=axlab_dict)
ax1.set_xlabel('Provincias', fontdict=axlab_dict)
ax1.set_title('Diagrama de Pareto - produccion por provincia', fontdict=axtitle_dict)

# Crear un segundo eje para la línea de acumulación
ax2 = ax1.twinx()
ax2.plot(prod_prov_2.provincia_nombre, prod_prov_2.acumulado, color='red', marker='o', linestyle='-', label='Acumulado (%)')
ax2.set_ylabel('Porcentaje acumulado', fontdict=axlab_dict)

# Añadir líneas de referencia al 80%
ax2.axhline(80, color='gray', linestyle='--', linewidth=1, label='80%')
plt.legend(loc='upper left')

# Ajustar el diseño
plt.tight_layout()
plt.show()

In [ ]:
f, axs = plt.subplots(1, 2, figsize=(15, 8))
sns.scatterplot(data=df, x="superficie_sembrada_ha", y="rendimiento_kgxha",
                hue="provincia_nombre", alpha=.5, ax=axs[0])
sns.histplot(data=df, x="provincia_nombre", hue="provincia_nombre", shrink=.9,
             alpha=.5, legend=True, ax=axs[1])
f.tight_layout()